# Market anomaly exploratory analysis

This notebook inspects the reproducible sample output generated by `python run_all.py --mode sample`. It validates the analysis contract, summarizes risk levels, and reviews the highest-scoring anomaly candidates.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("../data/processed/market_anomaly_results.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Run `python run_all.py --mode sample` from the project root first.")

market = pd.read_csv(DATA_PATH, parse_dates=["date"])
required = {"date", "symbol", "close", "anomaly_score", "model_anomaly", "risk_level"}
missing = sorted(required - set(market.columns))
if missing:
    raise ValueError(f"Analysis output is missing required columns: {missing}")

print(f"Rows: {len(market):,}; symbols: {market['symbol'].nunique():,}; period: {market['date'].min().date()} to {market['date'].max().date()}")

In [ ]:
risk_summary = (
    market.groupby("risk_level", dropna=False)
    .agg(rows=("date", "size"), anomalies=("model_anomaly", "sum"), mean_score=("anomaly_score", "mean"))
    .sort_values("mean_score", ascending=False)
)
risk_summary

In [ ]:
focus_symbol = str(market["symbol"].iloc[0])
focus = market.loc[market["symbol"].astype(str) == focus_symbol].sort_values("date")
flagged = focus.loc[focus["model_anomaly"] == 1]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(focus["date"], focus["anomaly_score"], color="#2563EB", linewidth=1.8, label="Anomaly score")
ax.scatter(flagged["date"], flagged["anomaly_score"], color="#B91C1C", marker="D", label="Model anomaly", zorder=3)
ax.axhline(0, color="#64748B", linewidth=0.8, linestyle="--")
ax.set(title=f"Anomaly score — {focus_symbol}", xlabel="Date", ylabel="Score")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
top_candidates = (
    market.sort_values("anomaly_score", ascending=False)
    .loc[:, ["date", "symbol", "close", "anomaly_score", "risk_level", "model_anomaly"]]
    .head(15)
)
top_candidates